# GPT-OSS-20B — Query2Doc Query Generator

**Experiment:** exp_009 — GPT-OSS-20B + Dense Retrieval
**Technique:** Query2Doc (pseudo-document generation)
**Reference baseline:** exp_003 (Qwen 2.5 3B, NDCG@10=0.5435)

## Model Details
- **Model:** `openai/gpt-oss-20b` (20.91B total, **3.61B active** per token)
- **Architecture:** MoE Transformer — 32 experts, top-4 routed per token (GQA, RoPE, SwiGLU, RMSNorm)
- **Developer:** OpenAI (first open-source model), August 2025
- **Training:** Trillions of tokens, **mostly English** (STEM, coding, general knowledge)
- **Vocab:** 201,088 tokens (o200k_harmony BPE via tiktoken, same base as GPT-4o)
- **Context:** 131,072 tokens (128K)
- **Quantization:** Native MXFP4 on MoE weights (4.25 bits/param, trained at this precision)
- **License:** Apache 2.0

## GPU Strategy: A100 (40 GB) — Unsloth BNB 4-bit
- **Loading:** Unsloth `load_in_4bit=True` (bitsandbytes NF4, avoids MXFP4 hardware issues)
- **VRAM:** ~14 GB model → ~26 GB free on A100 for batching
- **Batch size:** Start with 4, increase if VRAM allows
- **Estimated time:** ~30-60 min for 2,896 queries

## GPT-OSS-Specific Notes
- **CRITICAL: Harmony chat format is MANDATORY** — model will not work otherwise
- **Set `reasoning_effort="low"`** — minimizes chain-of-thought overhead for QE
- **Output parsing:** Extract `final` channel content, strip `analysis` reasoning tokens
- **Arabic is HIGH RISK:** English-dominant training, Arabic ILMAAM ~58%
- **MoE architecture:** Only 3.6B active params — may behave like a weakly-trained 3.6B for Arabic
- **Bleeding-edge deps:** Requires `torch>=2.8.0`, `triton>=3.4.0`, `transformers==4.56.2`

## Why This Experiment Matters
This is the **only MoE model** and the **only non-Arabic-specialized model** in our comparison.
- If it works: general-purpose MoE can do Arabic QE
- If it fails: confirms Arabic-specialized training is essential
- Either way: novel MoE vs dense comparison for Arabic QE

## Key Research Sources
- Model card: https://huggingface.co/openai/gpt-oss-20b
- Paper: arXiv:2508.10925
- Unsloth docs: https://unsloth.ai/docs/models/gpt-oss-how-to-run-and-fine-tune
- Full research: `research_decisions/gpt_oss_20b_research.md`

---

## Step 1: Install Dependencies

> After this cell: Runtime -> Restart runtime, then continue from Step 2.

In [ ]:
# ── Step 1: Install all dependencies ──────────────────────────────────────────
#
# Runtime: A100 (40 GB) — select in Runtime → Change runtime type → A100
# GPT-OSS-20B via Unsloth BNB 4-bit: ~14 GB VRAM.
# T4 (15 GB) is possible but very tight — A100 recommended.
#
# GPT-OSS requires bleeding-edge packages:
#   - torch >= 2.8.0
#   - triton >= 3.4.0
#   - transformers == 4.56.2
#   - unsloth (latest from GitHub)
#
# Java is required by pyserini (for MIRACL data loading).
#
# After this cell: Runtime → Restart runtime, then continue from Step 2.
# ──────────────────────────────────────────────────────────────────────────────

# 1. Java (required by pyserini — hidden dependency in MIRACLDataLoader)
!apt-get install -qq openjdk-21-jdk-headless

# 2. Retrieval / data loading libraries
!pip install -q pyserini faiss-cpu

# 3. Unsloth + dependencies (from official GPT-OSS notebook)
#    This installs torch >= 2.8.0, triton >= 3.4.0, transformers, bitsandbytes
!pip install "torch>=2.8.0" "triton>=3.4.0" torchvision bitsandbytes "transformers==4.56.2"
!pip install "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo"
!pip install "unsloth[base] @ git+https://github.com/unslothai/unsloth"

# 4. Triton kernels (required for MoE routing on non-H100 GPUs)
!pip install git+https://github.com/triton-lang/triton.git@0add68262ab0a2e33b84524346cb27cbb2787356#subdirectory=python/triton_kernels

# 5. ML / utility libraries
!pip install -q datasets accelerate tqdm

print("\n" + "=" * 60)
print("Installation complete")
print("=" * 60)
print("IMPORTANT: Restart runtime now!")
print("   Runtime -> Restart runtime")
print("   Then run cells starting from Step 2")
print("=" * 60)

## Step 2: Mount Drive and Setup Environment

> Run this after restarting runtime

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Clone project repo (or pull if already cloned)
!git clone https://github.com/Osmanoor/graduation.git 2>/dev/null || (cd /content/graduation && git pull)
%cd /content/graduation/arabic-rag-query-enhancement

import os
import sys

# Java home required by pyserini
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-21-openjdk-amd64'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

sys.path.insert(0, '/content/graduation/arabic-rag-query-enhancement')

# Verify Java
!java -version

import torch
print(f"\nEnvironment configured")
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu_name}")
    print(f"VRAM: {gpu_mem:.1f} GB")
    if gpu_mem < 20:
        print("WARNING: T4 detected. GPT-OSS-20B will be VERY tight (~14 GB).")
        print("  Consider switching to A100 for comfortable headroom.")

import transformers
print(f"Transformers version: {transformers.__version__}")
if transformers.__version__ < "4.55.0":
    print("WARNING: GPT-OSS requires transformers >= 4.55.0!")
    print("  Re-run Step 1 install or: pip install transformers==4.56.2")

## Step 3: Load MIRACL Arabic Data

In [ ]:
from src.utils.data_loader import MIRACLDataLoader

data_loader = MIRACLDataLoader(language="ar", split="dev")
topics, qrels = data_loader.load_all()

query_ids = list(topics.keys())
query_texts = [topics[qid]['title'] for qid in query_ids]

print(f"\nDataset Statistics:")
print(f"  Queries: {len(query_ids)}")
print(f"  Qrels:   {len(qrels)}")
print(f"\nSample query: {query_texts[0]}")

## Step 4: Initialize GPT-OSS-20B via Unsloth

**Strategy:** Unsloth `load_in_4bit=True` (bitsandbytes NF4)
- Avoids MXFP4 hardware compatibility issues on T4/A100
- Most tested loading path for Colab
- `FastLanguageModel.for_inference()` gives 2x speedup

**Architecture:** MoE Transformer (32 experts, top-4 per token)
- 20.91B total params, but only 3.61B ACTIVE per token
- Standard Transformer routing — batching should work
- Alternating sliding window (128 tokens) + full attention

**CRITICAL:** Harmony chat format is mandatory. `apply_chat_template()` handles this.
Set `reasoning_effort="low"` to minimize chain-of-thought overhead.

In [ ]:
import torch
from unsloth import FastLanguageModel

MODEL_NAME = "unsloth/gpt-oss-20b"

# ── Configuration ─────────────────────────────────────────────────────────────
# GPT-OSS-20B via Unsloth BNB 4-bit: ~14 GB VRAM.
# MoE routing may add memory overhead — start conservative with batch_size.
# A100: try 4-8. T4: try 1-2.
BATCH_SIZE = 4  # A100: start with 4, increase if VRAM allows. T4: use 1.
REASONING_EFFORT = "low"  # "low", "medium", "high" — low minimizes CoT overhead
# ──────────────────────────────────────────────────────────────────────────────

print(f"Loading {MODEL_NAME}...")
print(f"Mode: Unsloth BNB 4-bit (avoids MXFP4 hardware issues)")
print(f"Target batch size: {BATCH_SIZE}")
print(f"Reasoning effort: {REASONING_EFFORT}")

# Load model via Unsloth
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    dtype=None,  # auto-detect (BF16 on A100, FP16 on T4)
    max_seq_length=1024,  # sufficient for query + 128 generated tokens
    load_in_4bit=True,
    full_finetuning=False,
)

# Enable Unsloth's 2x inference speedup
FastLanguageModel.for_inference(model)

# Set padding for batch generation
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'left'  # Required for decoder-only batch generation

# Report VRAM usage
if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info()
    used = (total - free) / 1e9
    print(f"\nGPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {used:.1f} GB used / {total/1e9:.1f} GB total")
    print(f"Free: {free/1e9:.1f} GB")
    # Suggest batch size (conservative for MoE — experts add overhead)
    if free / 1e9 > 25:
        suggested_bs = 8
    elif free / 1e9 > 15:
        suggested_bs = 4
    elif free / 1e9 > 5:
        suggested_bs = 2
    else:
        suggested_bs = 1
    print(f"Suggested batch size: {suggested_bs} (conservative for MoE overhead)")
    if suggested_bs != BATCH_SIZE:
        print(f"  -> Consider changing BATCH_SIZE to {suggested_bs}")

print(f"\nGPT-OSS-20B ready (Unsloth BNB 4-bit)")
print(f"  Total params: 20.91B (3.61B active per token)")
print(f"  Experts: 32 total, top-4 routed")
print(f"  Vocab size: {len(tokenizer)}")
print(f"  Pad token: {tokenizer.pad_token}")

import transformers
print(f"  Transformers: {transformers.__version__}")

## Step 5: Sanity Check — First 5 Queries (NO REASONING)

**This is the MOST IMPORTANT step for GPT-OSS-20B.**

**FIX: Force generation to start in the `final` channel.**
By appending `<|start|>assistant<|channel|>final<|message|>` as the generation prefix,
the model skips the `analysis` channel entirely — no reasoning, no English CoT, much faster.

Arabic quality is the biggest unknown. Check carefully:
- Is the output in Arabic? (English-dominant model may respond in English)
- Is the content relevant to the query?
- Are there harmony format artifacts in the decoded text?
- Expansion ratio reasonable (5-12x)?

**If output is in English or garbage:** Try stronger Arabic system prompt (Step 5b).
**If still fails:** Document as a finding and stop.

In [ ]:
import re

SYSTEM_PROMPT = (
    "You are asked to write a passage that answers the given query. "
    "Do not ask the user for further clarification. "
    "Respond in Arabic only."
)

# Sampling parameters — match cross-model comparison settings
TEMPERATURE = 0.7
MAX_NEW_TOKENS = 128
TOP_P = 0.9

# ── Force final channel (skip reasoning entirely) ────────────────────────────
# GPT-OSS uses harmony format with channels: analysis (CoT), commentary, final.
# By default, the model reasons in the analysis channel first (slow, English).
# We force generation to start directly in the "final" channel.
FINAL_CHANNEL_PREFIX = "<|start|>assistant<|channel|>final<|message|>"
# ──────────────────────────────────────────────────────────────────────────────

# Harmony tokens to strip from decoded output
HARMONY_PATTERN = re.compile(
    r'<\|start\|>|<\|end\|>|<\|message\|>|<\|channel\|>|'
    r'<\|return\|>|<\|startoftext\|>|<\|endoftext\|>|'
    r'<\|call\|>|<\|result\|>'
)

def clean_output(text):
    """Clean generated text: strip harmony artifacts, channel markers, whitespace."""
    # If analysis channel leaked through despite prefix forcing, extract final content
    if '<|channel|>final<|message|>' in text:
        text = text.split('<|channel|>final<|message|>')[-1]
    # Strip all harmony special tokens
    text = HARMONY_PATTERN.sub('', text)
    # Strip any channel labels that appeared as plain text
    text = re.sub(r'\b(analysis|commentary|final)\b', '', text)
    # Clean up whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def generate_single_forced_final(query):
    """Generate pseudo-document, forcing the model into the final channel.
    Skips analysis/reasoning entirely — faster and avoids English CoT.
    """
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": query}
    ]

    # Build prompt WITHOUT generation prompt (we'll add our own)
    prompt_ids = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=False,
        return_tensors="pt",
        reasoning_effort=REASONING_EFFORT
    ).to(model.device)

    # Encode forced final-channel prefix
    prefix_ids = tokenizer.encode(FINAL_CHANNEL_PREFIX, add_special_tokens=False)
    prefix_tensor = torch.tensor([prefix_ids], device=model.device)

    # Concatenate: [prompt] + [forced final channel prefix]
    full_input = torch.cat([prompt_ids, prefix_tensor], dim=-1)
    input_length = full_input.shape[-1]

    with torch.no_grad():
        outputs = model.generate(
            full_input,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id
        )

    # Decode only the NEW tokens (after our full input)
    generated = tokenizer.decode(
        outputs[0][input_length:],
        skip_special_tokens=True
    ).strip()

    return clean_output(generated)


print("Sanity check: testing on first 5 queries (FORCED FINAL CHANNEL — no reasoning)\n")
print(f"Sampling: temp={TEMPERATURE}, top_p={TOP_P}")
print(f"Prefix: {FINAL_CHANNEL_PREFIX}")
print(f"System prompt: {SYSTEM_PROMPT}")
print("=" * 60)

for i in range(5):
    query = query_texts[i]
    generated = generate_single_forced_final(query)

    # Construct enhanced query (Query2Doc format: original + pseudo-doc)
    enhanced = f"{query} {generated}"
    ratio = len(enhanced) / max(len(query), 1)

    # Detect language issues
    arabic_chars = sum(1 for c in generated if '\u0600' <= c <= '\u06FF')
    latin_chars = sum(1 for c in generated if c.isascii() and c.isalpha())
    total_alpha = arabic_chars + latin_chars
    arabic_pct = (arabic_chars / max(total_alpha, 1)) * 100

    lang_warn = ""
    if arabic_pct < 50:
        lang_warn = f" *** WARNING: Only {arabic_pct:.0f}% Arabic! ***"
    elif arabic_pct < 80:
        lang_warn = f" (Mixed: {arabic_pct:.0f}% Arabic)"

    print(f"\nQuery {i+1} [{query_ids[i]}]: {query}")
    print(f"Generated ({len(generated)} chars, {arabic_pct:.0f}% Arabic):{lang_warn}")
    print(f"{generated[:300]}..." if len(generated) > 300 else generated)
    print(f"Expansion ratio: {ratio:.1f}x")

print("\n" + "=" * 60)
print("SANITY CHECK COMPLETE")
print("\nBefore proceeding, verify:")
print("  [ ] Output is in Arabic (>80% Arabic characters)")
print("  [ ] No 'analysis' reasoning text in output")
print("  [ ] Content is relevant to query topic")
print("  [ ] Expansion ratio 3-15x")
print("  [ ] Much faster than before (no CoT overhead)")
print("\nIf Arabic quality is poor:")
print("  1. Try Step 5b (stronger Arabic prompt)")
print("  2. If still bad, document as finding and STOP")

## Step 5b: (Optional) Retry with Stronger Arabic Prompt

**Only run this if Step 5 output was mostly English or mixed language.**

Adds explicit Arabic instruction in Arabic + English to maximize compliance.

In [ ]:
# ── OPTIONAL: Stronger Arabic prompt ──────────────────────────────────────────
# Only run this cell if Step 5 showed poor Arabic output.
# This replaces SYSTEM_PROMPT with a bilingual version.
# Still uses forced final channel (no reasoning).
# ──────────────────────────────────────────────────────────────────────────────

SYSTEM_PROMPT_STRONG = (
    "أنت مطلوب منك كتابة فقرة تجيب على الاستعلام المعطى. "
    "لا تطلب من المستخدم توضيحاً إضافياً. "
    "أجب باللغة العربية فقط.\n\n"
    "You are asked to write a passage that answers the given query. "
    "Do not ask the user for further clarification. "
    "You MUST respond in Arabic only. Do NOT use English."
)

print("RETRY: Testing with stronger Arabic prompt (still forced final channel)\n")
print(f"System prompt: {SYSTEM_PROMPT_STRONG[:80]}...")
print("=" * 60)

# Temporarily swap system prompt
_original_prompt = SYSTEM_PROMPT
SYSTEM_PROMPT = SYSTEM_PROMPT_STRONG

for i in range(5):
    query = query_texts[i]
    generated = generate_single_forced_final(query)

    arabic_chars = sum(1 for c in generated if '\u0600' <= c <= '\u06FF')
    latin_chars = sum(1 for c in generated if c.isascii() and c.isalpha())
    total_alpha = arabic_chars + latin_chars
    arabic_pct = (arabic_chars / max(total_alpha, 1)) * 100
    ratio = len(f"{query} {generated}") / max(len(query), 1)

    print(f"\nQuery {i+1}: {query}")
    print(f"Generated ({len(generated)} chars, {arabic_pct:.0f}% Arabic):")
    print(f"{generated[:300]}..." if len(generated) > 300 else generated)
    print(f"Expansion ratio: {ratio:.1f}x")

# Restore original prompt (user decides which to keep)
SYSTEM_PROMPT = _original_prompt

print("\n" + "=" * 60)
print("If Arabic is now >80%, update SYSTEM_PROMPT in Step 5 cell to use")
print("  SYSTEM_PROMPT_STRONG, then proceed to Step 6.")
print("If still poor: Document as thesis finding and STOP.")

## Step 6: Full Generation — 2,896 Queries (Forced Final Channel)

**Mode:** Batched generation with forced `final` channel prefix
**Key fix:** Appending `<|start|>assistant<|channel|>final<|message|>` to each prompt
skips the analysis/reasoning channel entirely — much faster, no English CoT leakage.

**MoE note:** Standard Transformer routing — batching should work, but MoE expert
activation adds memory overhead. OOM fallback to single-query mode included.
**Expected time:** ~30-60 min on A100 with batch_size=4
**Checkpoints:** Saves progress every 200 queries to pkl

In [ ]:
import time
import pickle
from tqdm.notebook import tqdm

CHECKPOINT_EVERY = 200
CHECKPOINT_PATH = 'enhanced_queries_gpt_oss_20b_checkpoint.pkl'

# Pre-encode the forced final-channel prefix (reused for every query)
FINAL_PREFIX_IDS = tokenizer.encode(FINAL_CHANNEL_PREFIX, add_special_tokens=False)


def generate_batch_forced_final(batch_queries):
    """Generate pseudo-documents for a batch of queries.
    Forces each prompt into the 'final' channel — skips analysis/reasoning.
    Uses left-padded tokenization for parallel generation.
    """
    # Build each prompt with forced final-channel suffix
    batch_input_ids = []
    for query in batch_queries:
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": query}
        ]
        # Get prompt IDs without generation prompt
        prompt_ids = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=False,
            reasoning_effort=REASONING_EFFORT
        )
        # Append forced final-channel prefix
        full_ids = prompt_ids + FINAL_PREFIX_IDS
        batch_input_ids.append(full_ids)

    # Left-pad to equal length for batching
    max_len = max(len(ids) for ids in batch_input_ids)
    pad_id = tokenizer.pad_token_id
    padded = []
    for ids in batch_input_ids:
        padding = [pad_id] * (max_len - len(ids))
        padded.append(padding + ids)

    input_tensor = torch.tensor(padded, device=model.device)
    attention_mask = (input_tensor != pad_id).long()
    input_length = input_tensor.shape[1]

    with torch.no_grad():
        outputs = model.generate(
            input_ids=input_tensor,
            attention_mask=attention_mask,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            do_sample=True,
            pad_token_id=pad_id
        )

    # Decode only generated tokens (after our full input)
    generated_texts = tokenizer.batch_decode(
        outputs[:, input_length:],
        skip_special_tokens=True
    )

    # Clean and combine with original query
    enhanced = []
    for q, g in zip(batch_queries, generated_texts):
        g_clean = clean_output(g.strip())
        enhanced.append(f"{q} {g_clean}")
    return enhanced


print("=" * 60)
print(f"FULL RUN: GPT-OSS-20B (Unsloth BNB 4-bit, temp={TEMPERATURE})")
print(f"Queries: {len(query_texts)}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Mode: Batched (batch_size={BATCH_SIZE}), FORCED FINAL CHANNEL (no reasoning)")
print(f"Checkpoints: every {CHECKPOINT_EVERY} queries")
num_batches = (len(query_texts) + BATCH_SIZE - 1) // BATCH_SIZE
print(f"Estimated: {num_batches} batches")
print("=" * 60 + "\n")

start_time = time.time()
enhanced_queries = []
errors = []
arabic_pct_samples = []  # Track Arabic percentage for quality monitoring

# Process in batches
for batch_idx in tqdm(range(num_batches), desc="Enhancing queries"):
    start_idx = batch_idx * BATCH_SIZE
    end_idx = min(start_idx + BATCH_SIZE, len(query_texts))
    batch_queries = query_texts[start_idx:end_idx]
    batch_qids = query_ids[start_idx:end_idx]

    try:
        batch_enhanced = generate_batch_forced_final(batch_queries)
        enhanced_queries.extend(batch_enhanced)

        # Sample Arabic percentage every 50 batches for quality monitoring
        if batch_idx % 50 == 0 and batch_enhanced:
            sample = batch_enhanced[0]
            ar_chars = sum(1 for c in sample if '\u0600' <= c <= '\u06FF')
            lat_chars = sum(1 for c in sample if c.isascii() and c.isalpha())
            total = ar_chars + lat_chars
            if total > 0:
                arabic_pct_samples.append(ar_chars / total * 100)

    except RuntimeError as e:
        if "out of memory" in str(e).lower():
            # OOM: fall back to single-query for this batch
            print(f"\nOOM at batch {batch_idx}! Falling back to single-query mode...")
            torch.cuda.empty_cache()
            for j, (q, qid) in enumerate(zip(batch_queries, batch_qids)):
                try:
                    generated = generate_single_forced_final(q)
                    enhanced_queries.append(f"{q} {generated}")
                except Exception as e2:
                    print(f"  Error on query {start_idx+j} [{qid}]: {e2}")
                    errors.append((start_idx+j, qid, str(e2)))
                    enhanced_queries.append(q)  # Fallback to original
        else:
            # Non-OOM error: try single-query fallback
            print(f"\nError at batch {batch_idx}: {e}")
            print("Attempting single-query fallback...")
            torch.cuda.empty_cache()
            for j, (q, qid) in enumerate(zip(batch_queries, batch_qids)):
                try:
                    generated = generate_single_forced_final(q)
                    enhanced_queries.append(f"{q} {generated}")
                except Exception as e2:
                    print(f"  Error on query {start_idx+j} [{qid}]: {e2}")
                    errors.append((start_idx+j, qid, str(e2)))
                    enhanced_queries.append(q)

    # Checkpoint
    completed = len(enhanced_queries)
    if completed % CHECKPOINT_EVERY < BATCH_SIZE and completed > 0:
        with open(CHECKPOINT_PATH, 'wb') as f:
            pickle.dump({
                'enhanced_so_far': enhanced_queries,
                'completed': completed,
                'total': len(query_texts)
            }, f)

elapsed = time.time() - start_time
print(f"\nEnhanced {len(enhanced_queries)} queries in {elapsed/60:.1f} minutes")
print(f"  Speed: {len(enhanced_queries) / (elapsed/60):.1f} queries/minute")
print(f"  Batch size: {BATCH_SIZE}")
if errors:
    print(f"  Errors: {len(errors)} (fell back to original query)")
    for idx, qid, err in errors[:5]:
        print(f"    Query {idx} [{qid}]: {err}")
if arabic_pct_samples:
    import numpy as np
    print(f"\nArabic quality monitoring (sampled every 50 batches):")
    print(f"  Avg Arabic %: {np.mean(arabic_pct_samples):.1f}%")
    print(f"  Min Arabic %: {np.min(arabic_pct_samples):.1f}%")
    if np.mean(arabic_pct_samples) < 50:
        print("  *** WARNING: Low Arabic content! Results may be poor. ***")

## Step 7: Save Results

In [ ]:
import pickle
from datetime import datetime

data = {
    'query_ids': query_ids,
    'original': query_texts,
    'enhanced': enhanced_queries,
    'metadata': {
        'model': 'openai/gpt-oss-20b',
        'loading_path': MODEL_NAME,
        'architecture': 'MoE Transformer (32 experts, top-4 routed, GQA 64Q/8KV, RoPE+YaRN, SwiGLU, RMSNorm)',
        'total_params': '20.91B',
        'active_params': '3.61B per token',
        'developer': 'OpenAI',
        'training_data': 'Trillions of tokens, mostly English (STEM, coding, general)',
        'quantization': 'Unsloth BNB 4-bit (NF4)',
        'native_quantization': 'MXFP4 on MoE weights (trained at this precision)',
        'temperature': TEMPERATURE,
        'max_new_tokens': MAX_NEW_TOKENS,
        'top_p': TOP_P,
        'reasoning_effort': REASONING_EFFORT,
        'chat_format': 'Harmony (mandatory)',
        'batch_size': BATCH_SIZE,
        'gpu': torch.cuda.get_device_name(0),
        'technique': 'query2doc',
        'dataset': 'miracl-ar-dev',
        'date': datetime.now().isoformat(),
        'num_queries': len(query_ids),
        'runtime_minutes': round(elapsed / 60, 1),
        'queries_per_minute': round(len(enhanced_queries) / (elapsed / 60), 1),
        'errors': len(errors),
        'system_prompt': SYSTEM_PROMPT,
        'research_doc': 'research_decisions/gpt_oss_20b_research.md',
        'paper': 'arXiv:2508.10925',
        'arabic_quality_samples': arabic_pct_samples if arabic_pct_samples else 'not tracked'
    }
}

# Save locally in Colab
local_path = 'enhanced_queries_gpt_oss_20b.pkl'
with open(local_path, 'wb') as f:
    pickle.dump(data, f)
print(f"Saved locally: {local_path}")

# Save to Google Drive for persistence
drive_base = '/content/drive/MyDrive/graduation project/colab_data'
os.makedirs(drive_base, exist_ok=True)
drive_path = f'{drive_base}/enhanced_queries_gpt_oss_20b.pkl'
with open(drive_path, 'wb') as f:
    pickle.dump(data, f)
print(f"Saved to Drive: {drive_path}")

print(f"\nKey metadata:")
print(f"  Model: {data['metadata']['model']}")
print(f"  Active params: {data['metadata']['active_params']}")
print(f"  Quantization: {data['metadata']['quantization']}")
print(f"  Reasoning effort: {data['metadata']['reasoning_effort']}")
print(f"  Batch size: {data['metadata']['batch_size']}")
print(f"  Runtime: {data['metadata']['runtime_minutes']} min")
print(f"  Speed: {data['metadata']['queries_per_minute']} queries/min")
print(f"  Errors: {data['metadata']['errors']}")

## Step 8: Expansion Statistics

In [ ]:
import numpy as np

print("=" * 60)
print("EXPANSION STATISTICS")
print("=" * 60)

orig_lens = [len(q) for q in query_texts]
enh_lens = [len(q) for q in enhanced_queries]
ratios = [e / max(o, 1) for e, o in zip(enh_lens, orig_lens)]

# Arabic content analysis
arabic_pcts = []
for eq in enhanced_queries:
    ar = sum(1 for c in eq if '\u0600' <= c <= '\u06FF')
    lat = sum(1 for c in eq if c.isascii() and c.isalpha())
    total = ar + lat
    arabic_pcts.append(ar / max(total, 1) * 100)

print(f"\nGPT-OSS-20B (Unsloth BNB 4-bit, temp={TEMPERATURE}, reasoning={REASONING_EFFORT}):")
print(f"  Avg original length : {np.mean(orig_lens):.1f} chars")
print(f"  Avg enhanced length : {np.mean(enh_lens):.1f} chars")
print(f"  Avg expansion ratio : {np.mean(ratios):.2f}x")
print(f"  Median expansion    : {np.median(ratios):.2f}x")
print(f"  Min expansion       : {np.min(ratios):.2f}x")
print(f"  Max expansion       : {np.max(ratios):.2f}x")

print(f"\nArabic Content Analysis:")
print(f"  Avg Arabic %  : {np.mean(arabic_pcts):.1f}%")
print(f"  Min Arabic %  : {np.min(arabic_pcts):.1f}%")
print(f"  Queries <50%  : {sum(1 for p in arabic_pcts if p < 50)} / {len(arabic_pcts)}")
print(f"  Queries <80%  : {sum(1 for p in arabic_pcts if p < 80)} / {len(arabic_pcts)}")

if np.mean(arabic_pcts) < 50:
    print("\n*** WARNING: Average Arabic content below 50%! ***")
    print("  Model is likely generating mostly English text.")
    print("  This is an expected finding given English-dominant training.")
    print("  Document in thesis: GPT-OSS-20B Arabic QE = FAILED (language mismatch)")
elif np.mean(arabic_pcts) < 80:
    print("\n** NOTE: Mixed language output (50-80% Arabic) **")
    print("  Model produces some English terms mixed with Arabic.")
    print("  Retrieval quality may be affected by language mixing.")

print("\n" + "=" * 60)
print("REFERENCE (previous experiments):")
print("  exp_003 Qwen 2.5 3B:  Avg 9.73x (247.6 chars)")
print("  exp_005 Falcon-H1-3B: batch=1, temp=0.1")
print("  exp_006 Jais-2-8B:    Avg 10.46x (256.0 chars), 241.5 q/min")
print("  exp_007 Qwen3-4B:     232.6 q/min, batch=32")
print("  exp_008 ALLaM-7B:     DROPPED (-48.9% NDCG, tokenizer bug)")
print("=" * 60)

print(f"\nPkl file saved. Next step:")
print(f"  Open evaluate_enhanced_queries.ipynb")
print(f"  Upload {local_path} for Dense retrieval evaluation")

---

## Results (exp_009) — DROPPED

### Status: DROPPED (not evaluated on retrieval)

GPT-OSS-20B was dropped after sanity check due to two critical issues:

1. **Extreme inference slowness:** 71.4 seconds per batch of 4 queries on A100 = ~14 hours estimated for 2,896 queries. This is **70x slower** than Jais-2-8B (12 min total) and **~15x slower** than Falcon-H1-3B (batch=1). Root cause: MoE with 32 experts + BNB 4-bit quantization creates massive per-token routing overhead.

2. **Severe factual hallucinations:** 3 out of 5 sanity queries produced fluent Arabic text with completely wrong facts:
   - Query 3 (بطرس): Said Paul is "the Rock" — correct answer is Peter/بطرس
   - Query 5 (حرب الكونغو): Said Congo War was 1939-1945 — correct is 1996
   - Query 2 (الغواصات): Confused submarines with irrigation canals

### Sanity Check Results (after forced-final-channel fix)

| Query | Arabic % | Factual | Length | Notes |
|-------|----------|---------|--------|-------|
| Q1 | 100% | OK | 285 chars | Reasonable answer |
| Q2 | 100% | WRONG | — | Confused submarines with canals |
| Q3 | 100% | WRONG | — | Wrong biblical figure |
| Q4 | 100% | OK | — | Reasonable answer |
| Q5 | 100% | WRONG | — | Wrong war dates (1939-1945 vs 1996) |

### Key Technical Findings

| Finding | Details |
|---------|---------|
| **Analysis channel leak** | Default generation includes English CoT reasoning in `analysis` channel. Fixed by forcing `<\|start\|>assistant<\|channel\|>final<\|message\|>` prefix — skips reasoning entirely. |
| **Arabic compliance** | 100% Arabic after forced-final fix (was 12% with default reasoning). |
| **Speed** | ~17.9 sec/query on A100 (BNB 4-bit). Completely impractical for 2,896 queries. |
| **VRAM** | ~21.2 GB on A100 (40 GB). Model loads fine but routing overhead dominates latency. |
| **Hallucinations** | 60% factual error rate (3/5). Same pattern as ALLaM-7B — fluent Arabic, wrong facts. |

### Speed Comparison

| Model | Runtime | Speed | Factor vs mDPR baseline |
|-------|---------|-------|------------------------|
| Jais-2-8B (exp_006) | 12 min | 241.5 q/min | Best |
| Qwen3-4B (exp_007) | ~12 min | 232.6 q/min | ~Same |
| Qwen 2.5 3B (exp_003) | 40 min | ~72 q/min | 3x slower |
| Falcon-H1-3B (exp_005) | 60-90 min | ~48 q/min | 5x slower |
| ALLaM-7B (exp_008) | 16 min | ~181 q/min | 1.3x slower |
| **GPT-OSS-20B (exp_009)** | **~14h est.** | **~3.4 q/min** | **70x slower** |

### Verdict: DROP

Same conclusion as ALLaM-7B (exp_008): model produces fluent Arabic but factually unreliable content.
Additionally, MoE inference via BNB 4-bit is impractically slow for batch query expansion.

### Thesis Value (4 novel findings)
1. **MoE vs Dense for Arabic QE:** First empirical evidence that MoE routing overhead makes models like GPT-OSS impractical for batch Arabic QE tasks, even when only 3.6B params are active per token.
2. **English-dominant training:** Confirms that models trained primarily on English cannot reliably generate factual Arabic content, even with forced Arabic output.
3. **Forced-final-channel technique:** Novel approach to bypass Harmony chat format reasoning — achieved 100% Arabic output but couldn't fix factual accuracy.
4. **Hallucination pattern:** Same failure mode as ALLaM-7B — fluent language ≠ factual accuracy. Arabic-specialized training (Jais-2, Qwen) is essential.

---

## Lessons Learned

### Technical
1. **MoE + BNB 4-bit = extremely slow inference.** The 32-expert routing per token dominates latency despite only 3.6B active params. Not viable for batch tasks.
2. **Harmony chat format requires special handling.** Must force the `final` channel prefix to avoid English CoT reasoning in the `analysis` channel.
3. **`reasoning_effort` has no "off" option** — even "low" still produces analysis channel output. Forced-final-channel prefix is the only way to fully skip reasoning.
4. **Unsloth BNB 4-bit loading works** but doesn't solve the MoE routing overhead problem.

### Research
1. **English-dominant training → factual hallucinations in Arabic.** The model generates syntactically correct Arabic but invents facts (wrong dates, wrong people, wrong concepts).
2. **3/5 = 60% hallucination rate** — worse than random for retrieval augmentation.
3. **Arabic-specialized models are essential** for factual Arabic QE. General-purpose models (even 20B) fail.

### Key Finding for Thesis
GPT-OSS-20B is the only MoE model in our comparison. It demonstrates that:
- MoE architecture introduces prohibitive inference overhead for batch QE (~70x slower than best dense model)
- English-dominant training produces fluent but factually unreliable Arabic text
- Arabic-specialized training (Jais-2, Qwen) consistently outperforms larger general-purpose models
- **Model size ≠ Arabic QE quality:** 20.9B params (3.6B active) performed worse than 4B Qwen3 on all dimensions

---

## Citations

- Wang, L., Yang, N., & Wei, F. (2023). Query2doc: Query Expansion with Large Language Models. arXiv:2303.07678.
- OpenAI (2025). GPT-OSS. arXiv:2508.10925.
- Zhang, X., et al. (2023). MIRACL: A Multilingual Retrieval Dataset. TACL.
- Research notes: `research_decisions/gpt_oss_20b_research.md`